In [ ]:
import pandas as pd
import re

# Load HZ sheet
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines = pd.unique(hz[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', machine)
    if match:
        return int(match.group(1))
    return None

machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

# Convert tonnage column into list
def parse_tonnage(x):
    if pd.isna(x):
        return []
    return [int(t.strip()) for t in str(x).split(",")]

hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# Create compatibility matrix
matrix = []

for _, row in hz.iterrows():
    part = row["Part"]
    tonnage_list = row["Tonnage_List"]

    row_data = {"Part": part}

    for machine in machines:
        m_ton = machine_tonnage[machine]

        if m_ton in tonnage_list:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

compatibility_matrix = pd.DataFrame(matrix)

print(compatibility_matrix)

In [ ]:
import pandas as pd
import re

# Load VT sheet
vt = pd.read_excel(file_path, sheet_name="VT")

# Machine columns
machine_cols = [col for col in vt.columns if col.startswith("Machine")]

# Extract all machines
machines = pd.unique(vt[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', str(machine))
    if match:
        return int(match.group(1))
    return None

# Create machine → tonnage dictionary
machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

matrix = []

for _, row in vt.iterrows():

    part = row["Part"]
    part_tonnage = row["Tonnage"]

    row_data = {"Part": part}

    for machine in machines:

        if machine_tonnage[machine] == part_tonnage:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

vt_matrix = pd.DataFrame(matrix)

print(vt_matrix)

In [ ]:
import pandas as pd
import re

# Load HZ sheet
file_path = "C:/Users/Ex0164/Book1.xlsx"
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines = pd.unique(hz[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', machine)
    if match:
        return int(match.group(1))
    return None

machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

# Convert tonnage column into list
def parse_tonnage(x):
    if pd.isna(x):
        return []
    tonnages = []
    for t in str(x).split(","):
        t = t.strip()
        match = re.search(r'(\d+)', t)
        if match:
            tonnages.append(int(match.group(1)))
    return tonnages

hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# ==========================
# BUILD COMPATIBILITY MATRIX
# ==========================

part_machine_map = {}

for _, row in hz.iterrows():

    part = row["Part"]
    tonnage_list = row["Tonnage_List"]

    if part not in part_machine_map:
        part_machine_map[part] = {m:0 for m in machines}

    for machine in machines:

        m_ton = machine_tonnage[machine]

        if m_ton in tonnage_list:
            part_machine_map[part][machine] = 1

# Convert to dataframe
matrix_rows = []

for part, machine_dict in part_machine_map.items():
    row = {"Part":part}
    row.update(machine_dict)
    matrix_rows.append(row)

compatibility_matrix = pd.DataFrame(matrix_rows)

print(compatibility_matrix)

# Save compatibility matrix to Excel
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
compatibility_matrix.to_excel(output_path, index=False)

print(f"Compatibility matrix saved to {output_path}")

In [ ]:
import pandas as pd

# Files
vt_file = "C:/Users/Ex0164/Book1.xlsx"
unique_file = "C:/Users/Ex0164/unique_machines_vt.xlsx"

# Read VT sheet
vt = pd.read_excel(vt_file, sheet_name="VT")

# Read unique machines
unique = pd.read_excel(unique_file, sheet_name="Sheet1")

# Machine → tonnage from VT
machine_tonnage_vt = vt.set_index("Machine")["Tonnage"].to_dict()

# Machine → tonnage from unique machines file
machine_tonnage_unique = unique.set_index("Unique Machines")["Tonnage"].to_dict()

# Merge both mappings
machine_tonnage = {**machine_tonnage_vt, **machine_tonnage_unique}

# All machines
machines = list(machine_tonnage.keys())

matrix = []

for _, row in vt.iterrows():

    part = row["Part"]
    part_ton = row["Tonnage"]

    row_data = {"Part": part}

    for machine in machines:

        if machine_tonnage[machine] == part_ton:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

vt_matrix = pd.DataFrame(matrix)

# Save
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    vt_matrix.to_excel(writer, sheet_name="VT_Matrix", index=False)

print("Compatibility matrix updated with new machines")